# 02 Feature Exploration

目标：围绕互联网交易欺诈识别的风险特征体系，查看字段族、风险机制、缺失指示变量和候选变量池。


In [1]:
from pathlib import Path
import sys
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fraud_detection.utils import load_yaml
from fraud_detection.data import read_ieee_train, split_by_time
from fraud_detection.features import build_feature_system, candidate_features, add_missing_indicators, categorical_features

config = load_yaml(PROJECT_ROOT / "configs/base.yaml")
df = read_ieee_train(PROJECT_ROOT / config["data"]["raw_dir"])
train, valid, test, split_profile = split_by_time(df)
display(split_profile)


,Split,Rows,FraudRate,MinTime,MaxTime
0,train,413378,0.035169,86400,10437996
1,valid,88581,0.034341,10438003,13151840
2,test,88581,0.034804,13151880,15811131


In [2]:
features = candidate_features(train)
train2, valid2, test2, iv_features, indicators = add_missing_indicators(
    train, valid, test, features, config["features"]["high_missing_threshold"]
)
print("Candidate features:", len(features))
print("IV candidate features:", len(iv_features))
print("Missing indicators:", len(indicators))


Candidate features: 432
IV candidate features: 432
Missing indicators: 9


In [3]:
risk_system = build_feature_system(train2)
summary = risk_system.groupby(["FeatureCategory", "RiskMechanism"]).agg(
    FeatureCount=("Feature", "count"),
    AvgMissingRate=("MissingRate", "mean"),
).reset_index().sort_values("FeatureCount", ascending=False)
display(summary)
summary.to_csv(PROJECT_ROOT / "outputs/tables/feature_group_summary.csv", index=False, encoding="utf-8-sig")


,FeatureCategory,RiskMechanism,FeatureCount,AvgMissingRate
0,Vesta聚合统计特征,批量聚集与实体关联异常,339,0.426727
5,设备网络与数字指纹特征,设备网络与数字指纹异常,40,0.830470
3,行为时间差特征,行为时序节奏异常,15,0.591608
4,行为计数统计特征,批量聚集与实体关联异常,14,0.000000
2,支付工具与身份代理特征,支付身份与联系方式异常,12,0.226111
6,身份匹配一致性特征,身份一致性异常,9,0.557920
1,交易基础特征,交易场景异常,3,0.000000


In [4]:
cats = categorical_features(iv_features)
pd.DataFrame({"CategoricalFeature": cats}).to_csv(
    PROJECT_ROOT / "outputs/tables/categorical_feature_candidates.csv", index=False, encoding="utf-8-sig"
)
display(pd.DataFrame({"CategoricalFeature": cats}).head(40))


,CategoricalFeature
0,ProductCD
1,card1
2,card2
3,card3
4,card4
5,card5
6,card6
7,addr1
8,addr2
9,P_emaildomain
